### Step 1 — Load Prepared Data

In [1]:
import numpy
import pandas
print(numpy.__version__, pandas.__version__)


2.2.6 2.3.3


In [2]:
import pickle

domains = [
    "phishing",
    "fake_news",
    "political_statements",
    "product_reviews",
    "job_scams",
    "sms",
    "twitter_rumours"
]

all_domains_data = {}

for domain in domains:
    file_path = f"../data/prepared_domains/{domain}_processed.pkl"
    with open(file_path, "rb") as f:
        all_domains_data[domain] = pickle.load(f)

print("✅ All domains loaded successfully")


✅ All domains loaded successfully


### Step 2 — Load Backbone Model & Tokenizer

In [3]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForSequenceClassification

# Load backbone
backbone = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-small",
    num_labels=2
)




c:\Users\ASUS\Downloads\Fraud-Detection\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
for param in backbone.parameters():
    param.requires_grad = False

### Step 3 — Configure LoRA

In [5]:
for name, module in backbone.named_modules():
    if "attention" in name.lower():
        print(name, type(module))


deberta.encoder.layer.0.attention <class 'transformers.models.deberta_v2.modeling_deberta_v2.DebertaV2Attention'>
deberta.encoder.layer.0.attention.self <class 'transformers.models.deberta_v2.modeling_deberta_v2.DisentangledSelfAttention'>
deberta.encoder.layer.0.attention.self.query_proj <class 'torch.nn.modules.linear.Linear'>
deberta.encoder.layer.0.attention.self.key_proj <class 'torch.nn.modules.linear.Linear'>
deberta.encoder.layer.0.attention.self.value_proj <class 'torch.nn.modules.linear.Linear'>
deberta.encoder.layer.0.attention.self.pos_dropout <class 'torch.nn.modules.dropout.Dropout'>
deberta.encoder.layer.0.attention.self.dropout <class 'torch.nn.modules.dropout.Dropout'>
deberta.encoder.layer.0.attention.output <class 'transformers.models.deberta_v2.modeling_deberta_v2.DebertaV2SelfOutput'>
deberta.encoder.layer.0.attention.output.dense <class 'torch.nn.modules.linear.Linear'>
deberta.encoder.layer.0.attention.output.LayerNorm <class 'torch.nn.modules.normalization.Layer

In [6]:
# LoRA configuration with correct module names
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query_proj", "value_proj"],  # matches DeBERTa-v3 module names
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS"
)

### Step 4 — Create Domain-Specific LoRA Model

In [7]:
def init_domain_model():
    """
    Initialize and return a LoRA-wrapped DeBERTa-v3-base model
    ready for domain-specific fine-tuning.
    """
    domain_model = get_peft_model(backbone, lora_config)
    print("✅ Backbone + LoRA adapter ready")
    return domain_model

### Step 5 — Train Per Domain

In [8]:

print("Available domains:", list(all_domains_data.keys()))
print("Phishing DF columns:", all_domains_data["phishing"].columns)
# print("Phishing sample:\n", all_domains_data["phishing"].head())


Available domains: ['phishing', 'fake_news', 'political_statements', 'product_reviews', 'job_scams', 'sms', 'twitter_rumours']
Phishing DF columns: Index(['text', 'input_ids', 'attention_mask', 'url_count', 'email_count',
       'suspicious_keyword_count', 'has_suspicious_term', 'label'],
      dtype='object')


In [9]:
import torch
from torch.utils.data import Dataset

class DomainDataset(Dataset):
    def __init__(self, df):
        self.input_ids = torch.tensor(df["input_ids"].tolist(), dtype=torch.long)
        self.attention_mask = torch.tensor(df["attention_mask"].tolist(), dtype=torch.long)
        self.labels = torch.tensor(df["label"].tolist(), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }


In [10]:
from sklearn.model_selection import train_test_split

domain_name = "phishing"
df = all_domains_data[domain_name]

# Train (80%) vs temp (20%)
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# Validation (10%) vs Test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

print(f"""
Dataset split for '{domain_name}'
-------------------------------
Train:      {len(train_df)}
Validation: {len(val_df)}
Test:       {len(test_df)}
""")

train_dataset = DomainDataset(train_df)
val_dataset = DomainDataset(val_df)
test_dataset = DomainDataset(test_df)



Dataset split for 'phishing'
-------------------------------
Train:      12217
Validation: 1527
Test:       1528



C:\Users\ASUS\AppData\Local\Temp\ipykernel_23864\2837874868.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  self.input_ids = torch.tensor(df["input_ids"].tolist(), dtype=torch.long)


In [11]:
import torch
torch.set_num_threads(8)             # Adjust to number of CPU cores
torch.set_num_interop_threads(8)

In [12]:
model = init_domain_model()  # Your function wraps backbone with LoRA

✅ Backbone + LoRA adapter ready


In [13]:
from transformers import Trainer, TrainingArguments
training_args = TrainingArguments(
    output_dir=f"./outputs/{domain_name}",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    num_train_epochs=5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,

    dataloader_num_workers=0,      # <- important for Windows CPU
    gradient_accumulation_steps=2,
    fp16=False,
    gradient_checkpointing=False
)



In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

In [15]:
import torch
torch.set_num_threads(4)  # adjust based on CPU cores


In [ ]:
trainer.train()

c:\Users\ASUS\Downloads\Fraud-Detection\.venv\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Epoch,Training Loss,Validation Loss


### Step 6 — Save Domain Adapter

In [ ]:
model.save_pretrained(f"./adapters/{domain_name}")
print("🎉 LoRA adapter saved for", domain_name)


### Step 7 — Inference with Probability and Token-Level Explanation

In [ ]:
from peft import PeftModel
import torch.nn.functional as F

# Load shared backbone + domain adapter
model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-base", num_labels=2)
model = PeftModel.from_pretrained(model, "./adapters/phishing")

# Example usage
text = "Please verify your bank account immediately!"
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

with torch.no_grad():
    logits = model(**inputs).logits
    prob = F.softmax(logits, dim=-1)[0,1].item()
